# Reprodução do modelo isobárico: $B^{\pm}\to\pi^{\pm}\pi^+\pi^-$

Este notebook é o benchmark signal-only da análise LHCb publicada em [Phys. Rev. D 101, 012006](https://cds.cern.ch/record/2689374/files/1909.05212.pdf).

O objetivo inicial é avaliar os coeficientes cartesianos publicados e calcular as fit fractions por carga e as assimetrias $A_{CP}$. A reprodução estatística do ajuste experimental exige os eventos de dados da análise; este notebook valida a álgebra de amplitudes e a normalização do fitter.

## Convenções

Para cada componente, usamos a convenção da Tabela XXI:

$$c_q=(x+q\,\delta x)+i(y+q\,\delta y),\qquad q=+1\;(B^+),\;q=-1\;(B^-).$$

A contribuição $\rho(770)^0-\omega(782)$ é agrupada incluindo o termo de interferência entre os dois componentes.

In [1]:
import sys
from pathlib import Path

# Make the notebook work from the repository or from notebooks/benchmark.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "dalitzplotfitter").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break

import numpy as np
import pandas as pd

from dalitzplotfitter import (
    DecayChannel, DecayModel, PipiKKRescattering,
    RealImag, RelativisticBreitWigner, Resonance, RhoOmegaMixing,
    SigmaPole, enable_x64, ZemachP
)

enable_x64()

In [2]:
# O notebook é autocontido: não depende de um módulo Python externo.
# Massas/larguras em GeV. A fonte dos números está documentada na célula seguinte.
PAPER_MASS_WIDTHS = {
    "rho770": (0.7708, 0.1534),       # ajuste isobárico, Tabela XX
    "omega782": (0.78265, 0.00849),   # Laura++/PDG
    "f2_1270": (1.256, 0.1867),      # ajuste isobárico,
    "rho1450": (1.465, 0.400),        # Laura++/PDG
    "rho3_1690": (1.686, 0.186),      # Laura++/PDG
    "sigma": (0.563, 0.350),          # polo ajustado, Tabela XXII
}
COEFFICIENTS = {
    "rho770": (1.000, 0.000, -0.003, 0.000),
    "omega782": (0.091, -0.007, 0.000, -0.022),
    "f2_1270": (0.291, 0.204, -0.002, -0.179),
    "rho1450": (-0.223, 0.191, 0.031, 0.068),
    "rho3_1690": (0.073, -0.045, 0.044, -0.013),
    "rescattering": (0.142, -0.040, -0.047, -0.027),
    "sigma": (-0.485, 0.284, 0.231, 0.270),
}
PUBLISHED_FRACTIONS = {
    "+": {"rho770_omega782": 57.9, "f2_1270": 5.1, "rho1450": 6.2, "rho3_1690": 1.0, "rescattering": 0.8, "sigma": 22.2},
    "-": {"rho770_omega782": 53.3, "f2_1270": 12.6, "rho1450": 4.3, "rho3_1690": 0.1, "rescattering": 2.0, "sigma": 27.9},
}
PUBLISHED_ACP = {"rho770": 0.7, "omega782": -4.8, "f2_1270": 46.8, "rho1450": -12.9, "rho3_1690": -80.1, "rescattering": 44.7, "sigma": 16.0}

def coefficient(name, charge):
    x, y, dx, dy = COEFFICIENTS[name]
    return complex(x + charge * dx, y + charge * dy)

def components(charge):
    c = {name: coefficient(name, charge) for name in COEFFICIENTS}
    rho_mass, rho_width = PAPER_MASS_WIDTHS["rho770"]
    omega_mass, omega_width = PAPER_MASS_WIDTHS["omega782"]
    f2_mass, f2_width = PAPER_MASS_WIDTHS["f2_1270"]
    rho1450_mass, rho1450_width = PAPER_MASS_WIDTHS["rho1450"]
    rho3_mass, rho3_width = PAPER_MASS_WIDTHS["rho3_1690"]
    sigma_mass, sigma_width = PAPER_MASS_WIDTHS["sigma"]
    return [
        # Os dois termos usam os mesmos fatores P-wave do rho; o termo omega
        # mantém sua própria massa/largura apenas dentro de R_omega(m).
        Resonance("rho770", (0, 2), RealImag(c["rho770"].real, c["rho770"].imag),
                  lineshape=RhoOmegaMixing(component="rho", rho_mass=rho_mass,
                                           rho_width=rho_width, omega_mass=omega_mass,
                                           omega_width=omega_width),
                  mass=rho_mass, width=rho_width, spin=1, angular=ZemachP(),
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("omega782", (0, 2), RealImag(c["omega782"].real, c["omega782"].imag),
                  lineshape=RhoOmegaMixing(component="omega", rho_mass=rho_mass,
                                           rho_width=rho_width, omega_mass=omega_mass,
                                           omega_width=omega_width),
                  mass=rho_mass, width=rho_width, spin=1, angular=ZemachP(),
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("f2_1270", (0, 2), RealImag(c["f2_1270"].real, c["f2_1270"].imag),
                  lineshape=RelativisticBreitWigner(), mass=f2_mass, width=f2_width,
                  spin=2, angular=ZemachP(), resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho1450", (0, 2), RealImag(c["rho1450"].real, c["rho1450"].imag),
                  lineshape=RelativisticBreitWigner(), mass=rho1450_mass, width=rho1450_width,
                  spin=1, angular=ZemachP(), resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho3_1690", (0, 2), RealImag(c["rho3_1690"].real, c["rho3_1690"].imag),
                  lineshape=RelativisticBreitWigner(), mass=rho3_mass, width=rho3_width,
                  spin=3, angular=ZemachP(), resonance_radius=4.0, parent_radius=4.0),
        # Rescattering não tem polo: 1.0--1.5 GeV é a janela da parametrização.
        Resonance("rescattering", (0, 2), RealImag(c["rescattering"].real, c["rescattering"].imag),
                  lineshape=PipiKKRescattering(), mass=1.0, width=0.0, spin=0,
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("sigma", (0, 2), RealImag(c["sigma"].real, c["sigma"].imag),
                  lineshape=SigmaPole(), mass=sigma_mass, width=sigma_width, spin=0,
                  angular=ZemachP(), resonance_radius=4.0, parent_radius=4.0),
    ]

def make_model(charge, resolution):
    pion = "pi+" if charge > 0 else "pi-"
    other = "pi-" if charge > 0 else "pi+"
    return DecayModel(DecayChannel("B+" if charge > 0 else "B-", (pion, pion, other)), components(charge), normalization_method="gauss-legendre", normalize_components=True)

## Massas e larguras usadas

A massa e a largura do $\rho(770)^0$ são os valores ajustados no isobárico (Tabela XX), e os parâmetros do polo $\sigma$ vêm da Tabela XXII. Para $\omega(782)$, $f_2(1270)$, $\rho(1450)^0$ e $\rho_3(1690)^0$, a Tabela III especifica as linhashapes mas não imprime os números: usamos os valores nominais do catálogo Laura++/PDG. O rescattering não recebe massa/largura de ressonância; sua parametrização é definida na janela $1{,}0<m<1{,}5$ GeV.

In [3]:
mass_widths = pd.DataFrame.from_dict(
    PAPER_MASS_WIDTHS, orient="index", columns=["mass [GeV]", "width [GeV]"]
)
mass_widths.index.name = "component"
mass_widths

,mass [GeV],width [GeV]
component,,
rho770,0.77080,0.15340
omega782,0.78265,0.00849
f2_1270,1.25600,0.18670
rho1450,1.46500,0.40000
rho3_1690,1.68600,0.18600
sigma,0.56300,0.35000


## Coeficientes cartesianos publicados

Os valores abaixo são os centrais da Tabela XXI. Os componentes de erro não são usados nesta primeira etapa.

In [4]:
coefficients = pd.DataFrame(
    COEFFICIENTS, index=["x", "y", "dx", "dy"]
).T
coefficients

,x,y,dx,dy
rho770,1.000,0.000,-0.003,0.000
omega782,0.091,-0.007,0.000,-0.022
f2_1270,0.291,0.204,-0.002,-0.179
rho1450,-0.223,0.191,0.031,0.068
rho3_1690,0.073,-0.045,0.044,-0.013
rescattering,0.142,-0.040,-0.047,-0.027
sigma,-0.485,0.284,0.231,0.270


## Construção dos modelos $B^+$ e $B^-$

A resolução 180 é adequada para uma primeira execução. Para o resultado final de certificação, aumente a resolução e verifique a estabilidade numérica.

In [5]:
resolution = 180
plus_model = make_model(+1, resolution)
minus_model = make_model(-1, resolution)

print("B+ components:", [c.name for c in plus_model.amplitude_model.components])
print("B- components:", [c.name for c in minus_model.amplitude_model.components])

B+ components: ['rho770', 'omega782', 'f2_1270', 'rho1450', 'rho3_1690', 'rescattering', 'sigma']
B- components: ['rho770', 'omega782', 'f2_1270', 'rho1450', 'rho3_1690', 'rescattering', 'sigma']


In [6]:
def fractions_and_interference(model):
    cache = model._fraction_cache(None, None)
    fractions = np.asarray(cache.fit_fractions({})) * 100.0
    interference = np.asarray(cache.interference_fractions({})) * 100.0
    names = [component.name for component in cache.components]
    result = dict(zip(names, fractions, strict=True))
    result["rho770_omega782"] = (
        result["rho770"] + result["omega782"] + interference[0, 1]
    )
    return result, interference

plus_fractions, plus_interference = fractions_and_interference(plus_model)
minus_fractions, minus_interference = fractions_and_interference(minus_model)

## Fit fractions por carga

As Tabelas 12 e 13 apresentam o valor diagonal de cada componente. A soma das frações não precisa ser 100%, porque os termos de interferência não estão incluídos nas frações diagonais.

In [7]:
fraction_names = [
    "rho770_omega782", "f2_1270", "rho1450",
    "rho3_1690", "rescattering", "sigma",
]
fractions = pd.DataFrame({
    "B+ evaluator [%]": [plus_fractions[n] for n in fraction_names],
    "B+ paper [%]": [PUBLISHED_FRACTIONS["+"][n] for n in fraction_names],
    "B- evaluator [%]": [minus_fractions[n] for n in fraction_names],
    "B- paper [%]": [PUBLISHED_FRACTIONS["-"][n] for n in fraction_names],
}, index=fraction_names)
fractions["delta B+ [%]"] = fractions["B+ evaluator [%]"] - fractions["B+ paper [%]"]
fractions["delta B- [%]"] = fractions["B- evaluator [%]"] - fractions["B- paper [%]"]
fractions.round(4)

,B+ evaluator [%],B+ paper [%],B- evaluator [%],B- paper [%],delta B+ [%],delta B- [%]
rho770_omega782,57.2777,57.9,56.3970,53.3,-0.6223,3.0970
f2_1270,4.7148,5.1,12.9394,12.6,-0.3852,0.3394
rho1450,5.8242,6.2,4.4318,4.3,-0.3758,0.1318
rho3_1690,0.9555,1.0,0.1038,0.1,-0.0445,0.0038
rescattering,0.7572,0.8,1.9971,2.0,-0.0428,-0.0029
sigma,20.8118,22.2,28.5373,27.9,-1.3882,0.6373


## Frações de interferência

A matriz abaixo contém os termos $2\operatorname{Re}(A_iA_j^*)$ normalizados pela intensidade total. Ela é necessária para verificar a contribuição agrupada $\rho-\omega$ e a soma total do modelo.

In [8]:
component_names = [c.name for c in plus_model.amplitude_model.components]
plus_interference_table = pd.DataFrame(plus_interference, index=component_names, columns=component_names)
minus_interference_table = pd.DataFrame(minus_interference, index=component_names, columns=component_names)

print("B+ interference fractions [%]")
display(plus_interference_table.round(4))
print("B- interference fractions [%]")
display(minus_interference_table.round(4))

B+ interference fractions [%]


,rho770,omega782,f2_1270,rho1450,rho3_1690,rescattering,sigma
rho770,0.0000,1.0709,1.6989,8.2152,0.7559,0.5066,-1.1727
omega782,1.0709,0.0000,0.0026,0.0076,0.0007,-0.0003,0.0035
f2_1270,1.6989,0.0026,0.0000,0.3885,0.2038,0.1814,-0.1714
rho1450,8.2152,0.0076,0.3885,0.0000,0.1141,-0.2773,2.3772
rho3_1690,0.7559,0.0007,0.2038,0.1141,0.0000,0.0678,-0.3161
rescattering,0.5066,-0.0003,0.1814,-0.2773,0.0678,0.0000,-2.9272
sigma,-1.1727,0.0035,-0.1714,2.3772,-0.3161,-2.9272,0.0000


B- interference fractions [%]


,rho770,omega782,f2_1270,rho1450,rho3_1690,rescattering,sigma
rho770,0.0000,-0.0550,1.4146,2.0668,0.2141,1.1947,-5.8703
omega782,-0.0550,0.0000,0.0059,0.0066,-0.0005,-0.0002,0.0009
f2_1270,1.4146,0.0059,0.0000,0.8927,0.0144,0.3749,-1.1131
rho1450,2.0668,0.0066,0.8927,0.0000,-0.0430,-0.3777,2.0065
rho3_1690,0.2141,-0.0005,0.0144,-0.0430,0.0000,0.0171,-0.2044
rescattering,1.1947,-0.0002,0.3749,-0.3777,0.0171,0.0000,-5.0065
sigma,-5.8703,0.0009,-1.1131,2.0065,-0.2044,-5.0065,0.0000


## Assimetria $A_{CP}$

Nesta implementação usamos

$$A_{CP} = \frac{F_{B^-}-F_{B^+}}{F_{B^-}+F_{B^+}}.$$

O sinal deve ser confirmado junto à definição operacional usada na tabela do paper antes de transformar esta comparação em um teste de regressão.

In [9]:
acp_names = list(PUBLISHED_ACP)
acp = pd.DataFrame({
    "evaluator [%]": [
        100.0 * (minus_fractions[n] - plus_fractions[n]) /
        (minus_fractions[n] + plus_fractions[n])
        for n in acp_names
    ],
    "paper [%]": [PUBLISHED_ACP[n] for n in acp_names],
}, index=acp_names)
acp["delta [%]"] = acp["evaluator [%]"] - acp["paper [%]"]
acp.round(4)

,evaluator [%],paper [%],delta [%]
rho770,0.2534,0.7,-0.4466
omega782,-3.8405,-4.8,0.9595
f2_1270,46.5872,46.8,-0.2128
rho1450,-13.5764,-12.9,-0.6764
rho3_1690,-80.4062,-80.1,-0.3062
rescattering,45.0160,44.7,0.3160
sigma,15.6547,16.0,-0.3453


## Próxima etapa: toy closure

Depois que as diferenças acima forem explicadas pelas convenções de linha de forma, o próximo bloco deve gerar toys de sinal para as duas cargas e ajustar os coeficientes. O teste de certificação será:

1. injetar os coeficientes da Tabela XXI;
2. gerar amostras $B^+$ e $B^-$;
3. ajustar os coeficientes;
4. verificar recuperação dentro das incertezas estatísticas;
5. recalcular fit fractions e $A_{CP}$.

Isso valida o fitter com sinal puro, mas não substitui a reprodução do ajuste experimental com eficiência e background.